# Underwater enhancement comparison for Coralscapes segmentation

Continues the workflow in `coral_eval.ipynb`. It tests whether open-source underwater enhancement improves the performance of the **pretrained Coralscapes SegFormer** on the expert CVAT annotations of the Myora Reef BlueROV2 frames.

Input variants:
- **Raw passthrough**: no enhancement (baseline)
- **FUnIE-GAN**: learned enhancement (conditional GAN)
- **Sea-thru**: physics-based enhancement using an estimated depth map
- **Classical**: grey-world white balance, edge-redness reduction, UDCP dehazing and CLAHE
- **Resolution control**: the raw frame downscaled to 640 px and scaled back up, which separates the effect of colour correction from the effect of lost detail

For each variant, the same Coralscapes model predicts a segmentation mask. Predictions are mapped to the project classes used by the CVAT masks, then compared against the annotated pixels using per-class IoU, precision, recall, mean IoU and labelled-pixel accuracy.

**Pipeline (matches `coral_eval.ipynb`):** the segmentation model is loaded and run exactly as in the official Coralscapes approach (`SegformerImageProcessor` and plain `from_pretrained`, short side 1024, 1024x1024 sliding window, fp32, via `engine.load_reference` and `engine.segment_image`). Predictions are mapped to the project classes with the **name-based mapping in `configs/config.yaml`**. Only the input image changes between variants.

**Fairness safeguards:**
- the resolution control isolates the effect of downscaling from the effect of colour correction
- totals only use frames that *every* method processed
- a paired bootstrap over frames gives confidence intervals for each method versus raw

**Important interpretation:** this evaluates the effect of enhancement on segmentation against the available annotations, not visual enhancement quality in isolation. Only annotated pixels contribute to the metrics. With a small annotated set or classes absent from the annotations, results are noisy. The Coralscapes-to-project mapping is explicit and editable in the config; unassigned Coralscapes classes should not be treated as equivalent to any project class.

## Configure paths and run options

The CVAT export must already have been converted by `coral_eval.ipynb` (section "Convert the CVAT export"), which creates `work/own/images`, `work/own/masks` and `work/own/summary.json`. This notebook reads those files, so `coral_eval.ipynb` has to be run first. The notebook expects to run from the `zeroshot/` folder, and the shared modules are imported from `../utils`.

In [ ]:
from pathlib import Path
import os, sys, re, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

# Project and data paths
PROJECT_DIR = Path(".").expanduser().resolve()
os.chdir(PROJECT_DIR)      # config paths such as configs/classes_coralscapes_v1.json are relative to this folder
CONFIG_PATH = PROJECT_DIR / "configs" / "config.yaml"
WORK_DIR = (PROJECT_DIR / "work").resolve()      # the same folder coral_eval.ipynb writes to
OUTPUT_DIR = WORK_DIR / "enhancement_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Runtime controls
MAX_FRAMES = None       # None = all prepared annotated frames; set e.g. 3 for a quick smoke test
SHOW_PREVIEWS = True
SAVE_ENHANCED_IMAGES = True
SAVE_PREDICTION_MASKS = True
SEATHRU_MAX_SIZE = 640  # Sea-thru utility default; larger can be much slower
RESOLUTION_CONTROL = True   # also score "raw downscaled to SEATHRU_MAX_SIZE and back" (separates blur from colour correction)
N_BOOT = 2000               # bootstrap resamples (over frames) for the paired comparison

def safe(name):
    """Filesystem-safe version of a method name."""
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")

# Shared modules live in the repository's utils folder
sys.path.append("../utils")
from labels import load_config, pick_mapping, resolve_id2label
from engine import load_reference, pick_device, segment_image

cfg = load_config(str(CONFIG_PATH))
PROJECT_CLASSES = list(cfg["classes"]["names"])
IGNORE_INDEX = int(cfg["classes"]["ignore_index"])
OWN_DIR = Path(cfg["data"]["own_dir"])
if not OWN_DIR.is_absolute():
    OWN_DIR = (PROJECT_DIR / OWN_DIR).resolve()

print("Project:", PROJECT_DIR)
print("Work dir:", WORK_DIR)
print("Prepared annotated data:", OWN_DIR)
print("Project classes:", PROJECT_CLASSES)
print("Ignore index:", IGNORE_INDEX)

## Load annotated frames

The prepared mask images contain the integer class indices generated by `utils/prepare_cvat_export.py`, with unannotated pixels set to the configured ignore index. Each image and mask are paired by frame stem.

In [ ]:
summary_path = OWN_DIR / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        f"{summary_path} was not found. Run the 'Convert the CVAT export' section of "
        "coral_eval.ipynb first."
    )
kept_stems = json.loads(summary_path.read_text())["kept"]
if MAX_FRAMES is not None:
    kept_stems = kept_stems[:MAX_FRAMES]

def find_frame_image(stem):
    matches = [p for p in (OWN_DIR / "images").glob(f"{stem}.*") if p.is_file()]
    if not matches:
        raise FileNotFoundError(f"No prepared image found for {stem}")
    return matches[0]

def load_rgb(stem):
    return Image.open(find_frame_image(stem)).convert("RGB")

def load_gt(stem):
    p = OWN_DIR / "masks" / f"{stem}.png"
    if not p.exists():
        raise FileNotFoundError(f"Prepared mask not found: {p}")
    return np.asarray(Image.open(p)).astype(np.uint8)

print(f"Prepared frames available: {len(kept_stems)}")
print("Example stems:", kept_stems[:10])

## Load the segmentation model and map its classes onto the project classes

Loaded and mapped exactly as in `coral_eval.ipynb`: the checkpoint in `configs/config.yaml`, the class names it reports, and the **name-based** mapping section named by `model.mapping_key` (`v1_coralscapes_mapping` for V1, `coralscapes_mapping` for V2). Sand and rubble are separate classes. A model class that is not in the mapping becomes `unmapped` and counts as wrong on any annotated pixel.

The printed mapping should be reviewed before the results are trusted.

In [ ]:
import torch

# Enhancement modules (all in ../utils):
# classical_utils.py : classical_enhance(PIL.Image, ...)
# funie_utils.py     : load_funie_generator(...), enhance_frame(generator, image, device=...)
# seathru_utils.py   : load_seathru(device), enhance_frame(image, depth_bundle, device, ...)
# Segmentation goes through engine.py, shared with coral_eval.ipynb.
import classical_utils as classical_mod
import funie_utils as funie_mod
import seathru_utils as seathru_mod

# Enhancers run on CUDA when available, otherwise CPU; segmentation uses the device pick cuda > mps > cpu.
device = "cuda" if torch.cuda.is_available() else "cpu"
seg_device = pick_device()
print("Enhancement device:", device, "| segmentation device:", seg_device)

# Same loader, sliding-window inference and name-based mapping as coral_eval.ipynb.
coral_preprocessor, coral_model = load_reference(cfg["model"]["checkpoint"], seg_device,
                                                 id2label_file=cfg["model"].get("id2label_file"))
id2label, label_src = resolve_id2label(coral_model, cfg)
map_key, lut = pick_mapping(id2label, cfg, PROJECT_CLASSES)    # model id -> project class index (255 = unmapped)
print(f"Model: {cfg['model']['checkpoint']} | outputs: {coral_model.config.num_labels} | "
      f"class names from: {label_src} | mapping: '{map_key}'")
print("Preprocessor size:", coral_preprocessor.size)
for i, n in enumerate(PROJECT_CLASSES):
    srcs = [id2label[j] for j in sorted(id2label) if lut[j] == i]
    print(f"  {n:15s} <- {', '.join(srcs) if srcs else '(no model class)'}")
print(f"  {sum(lut[j] == 255 for j in id2label)} model classes are unmapped (count as wrong on labelled pixels)")

## Load enhancement models

Loads the pretrained weights for FUnIE-GAN and Sea-thru's Depth Anything V2 model. The first run downloads the model weights and requires internet access. The classical pipeline has no weights.

If a model cannot load because a dependency or download is unavailable, the cell raises the underlying error. That issue has to be fixed before a comparison can be treated as complete.

In [ ]:
import torch

t0 = time.time()
funie_generator = funie_mod.load_funie_generator(
    weights_path=str(WORK_DIR / "funie_generator.pth"), device=device
)
print(f"FUnIE-GAN loaded in {time.time()-t0:.1f}s")

t0 = time.time()
seathru_depth_bundle = seathru_mod.load_seathru(device)
print(f"Sea-thru depth model loaded in {time.time()-t0:.1f}s")

## Build enhancement functions and inspect one example

Each method receives the same original RGB frame and returns an RGB PIL image at the original dimensions. Sea-thru uses the utility's default assumed metric range derived from monocular relative depth, so its output is exploratory rather than a calibrated reconstruction.

In [ ]:
def enhance_raw(pil_image):
    return pil_image.convert("RGB")

def enhance_funie(pil_image):
    return funie_mod.enhance_frame(funie_generator, pil_image.convert("RGB"), device=device)

def enhance_seathru(pil_image):
    return seathru_mod.enhance_frame(
        pil_image.convert("RGB"),
        depth_bundle=seathru_depth_bundle,
        device=device,
        max_size=SEATHRU_MAX_SIZE,
        seed=0,
    )

def enhance_classical(pil_image):
    return classical_mod.classical_enhance(pil_image.convert("RGB"))

ENHANCERS = {
    "raw": enhance_raw,
    "FUnIE-GAN": enhance_funie,
    "Sea-thru": enhance_seathru,
    "Classical": enhance_classical,
}

# Control: no colour correction, but the same down/up-scaling Sea-thru goes through (longest side = SEATHRU_MAX_SIZE).
# If raw@640 scores clearly below raw, part of any Sea-thru deficit is lost detail, not the enhancement itself.
# (Check seathru_utils: if its max_size means something other than the longest side, adjust this function.)
CONTROL_NAME = f"raw@{SEATHRU_MAX_SIZE}px (control)"

def enhance_raw_downscaled(pil_image):
    img = pil_image.convert("RGB")
    s = SEATHRU_MAX_SIZE / max(img.size)
    if s >= 1:
        return img
    small = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.Resampling.BICUBIC)
    return small.resize(img.size, Image.Resampling.BICUBIC)

if RESOLUTION_CONTROL:
    ENHANCERS[CONTROL_NAME] = enhance_raw_downscaled

preview_stem = kept_stems[0]
preview_img = load_rgb(preview_stem)
fig, axes = plt.subplots(1, len(ENHANCERS), figsize=(4 * len(ENHANCERS), 4))
for ax, (method, fn) in zip(axes, ENHANCERS.items()):
    t0 = time.time()
    try:
        out = fn(preview_img)
        ax.imshow(out)
        ax.set_title(method)
        ax.axis("off")
        print(f"{method}: {out.size}, {time.time()-t0:.1f}s")
    except Exception as exc:
        ax.set_title(f"{method}\nFAILED")
        ax.axis("off")
        print(f"{method} failed: {type(exc).__name__}: {exc}")
plt.suptitle(f"Enhancement preview: {preview_stem}")
plt.tight_layout()
plt.show()

## Segmentation and metric functions

The exact same Coralscapes model and inference function are used for all variants. This isolates the effect of the input enhancement on the predictions as far as possible. Pixels with the configured ignore index in the ground truth are excluded. Predictions outside the explicit mapping are counted in an extra `unmapped` column and are incorrect for any annotated ground-truth class.

In [ ]:
N_CLASSES = len(PROJECT_CLASSES)
UNMAPPED = N_CLASSES

LUT_UNMAPPED = np.where(lut == 255, UNMAPPED, lut).astype(np.uint8)   # model id -> project class; unused -> UNMAPPED

def map_coralscapes_prediction(pred):
    return LUT_UNMAPPED[pred]

def confusion_for(gt, pred):
    valid = (gt != IGNORE_INDEX) & (gt < N_CLASSES)
    g = gt[valid].astype(np.int64)
    p = pred[valid].astype(np.int64)
    p = np.where((p >= 0) & (p <= UNMAPPED), p, UNMAPPED)
    return np.bincount(g * (N_CLASSES + 1) + p, minlength=N_CLASSES * (N_CLASSES + 1)).reshape(N_CLASSES, N_CLASSES + 1)

def metrics_from_confusion(conf):
    out = []
    for i, name in enumerate(PROJECT_CLASSES):
        tp = int(conf[i, i])
        gt_pixels = int(conf[i, :].sum())
        pred_pixels = int(conf[:, i].sum())
        fp = pred_pixels - tp
        fn = gt_pixels - tp
        denom_iou = tp + fp + fn
        out.append({
            "class": name,
            "gt_pixels": gt_pixels,
            "iou": tp / denom_iou if denom_iou else np.nan,
            "precision": tp / (tp + fp) if (tp + fp) else np.nan,
            "recall": tp / (tp + fn) if (tp + fn) else np.nan,
        })
    total = int(conf.sum())
    correct = int(np.trace(conf[:, :N_CLASSES]))
    present_ious = [r["iou"] for r in out if r["gt_pixels"] > 0 and np.isfinite(r["iou"])]
    summary = {
        "mIoU_present_classes": float(np.mean(present_ious)) if present_ious else np.nan,
        "labelled_pixel_accuracy": correct / total if total else np.nan,
        "labelled_pixels": total,
        "classes_present": len(present_ious),
    }
    return pd.DataFrame(out).set_index("class"), summary

def coral_predict(pil_image):
    """Sliding-window inference (same function coral_eval.ipynb uses)."""
    return segment_image(pil_image.convert("RGB"), coral_preprocessor, coral_model, seg_device)

## Run the comparison

Processes each annotated frame with each enhancement method, runs Coralscapes segmentation, and compares predictions to the corresponding CVAT mask. Intermediate enhanced images and mapped prediction masks can be saved for inspection. Progress is printed per frame and method.

Sea-thru is substantially slower than the other methods (about 9 s per 1080p frame, against 0.4 to 0.5 s for FUnIE-GAN and the classical pipeline). To test the workflow first, set `MAX_FRAMES = 2` in the options cell, then rerun the notebook from the top.

In [ ]:
frame_confs = {method: {} for method in ENHANCERS}      # method -> {stem: per-frame confusion}
frame_rows = []
method_errors = []

enhanced_dir = OUTPUT_DIR / "enhanced"
pred_dir = OUTPUT_DIR / "predictions"
if SAVE_ENHANCED_IMAGES:
    enhanced_dir.mkdir(parents=True, exist_ok=True)
if SAVE_PREDICTION_MASKS:
    pred_dir.mkdir(parents=True, exist_ok=True)

for frame_idx, stem in enumerate(kept_stems, start=1):
    original = load_rgb(stem)
    gt = load_gt(stem)
    if gt.shape != (original.height, original.width):
        raise ValueError(f"{stem}: mask shape {gt.shape} does not match image {(original.height, original.width)}")

    for method, enhance_fn in ENHANCERS.items():
        t0 = time.time()
        try:
            enhanced = enhance_fn(original)
            if enhanced.size != original.size:
                enhanced = enhanced.resize(original.size, Image.Resampling.BICUBIC)

            enhancement_seconds = time.time() - t0
            t1 = time.time()
            coral_pred = coral_predict(enhanced)
            segmentation_seconds = time.time() - t1
            mapped_pred = map_coralscapes_prediction(coral_pred)

            fc = confusion_for(gt, mapped_pred)
            frame_confs[method][stem] = fc
            _, metric_summary = metrics_from_confusion(fc)

            frame_rows.append({
                "frame": stem,
                "method": method,
                "labelled_pixel_accuracy": metric_summary["labelled_pixel_accuracy"],
                "mIoU_present_classes": metric_summary["mIoU_present_classes"],
                "enhancement_seconds": enhancement_seconds,
                "segmentation_seconds": segmentation_seconds,
                "total_seconds": enhancement_seconds + segmentation_seconds,
                "status": "ok",
            })

            if SAVE_ENHANCED_IMAGES:
                enhanced.save(enhanced_dir / f"{stem}__{safe(method)}.jpg", quality=95)
            if SAVE_PREDICTION_MASKS:
                Image.fromarray(mapped_pred).save(pred_dir / f"{stem}__{safe(method)}.png")

            print(
                f"[{frame_idx}/{len(kept_stems)}] {stem} | {method}: "
                f"acc={metric_summary['labelled_pixel_accuracy']:.3f}, "
                f"mIoU={metric_summary['mIoU_present_classes']:.3f}, "
                f"{enhancement_seconds + segmentation_seconds:.1f}s",
                flush=True,
            )
        except Exception as exc:
            method_errors.append({"frame": stem, "method": method, "error": repr(exc)})
            frame_rows.append({
                "frame": stem, "method": method,
                "labelled_pixel_accuracy": np.nan, "mIoU_present_classes": np.nan,
                "enhancement_seconds": np.nan, "segmentation_seconds": np.nan,
                "total_seconds": np.nan, "status": f"FAILED: {type(exc).__name__}: {exc}",
            })
            print(f"[{frame_idx}/{len(kept_stems)}] {stem} | {method} FAILED: {type(exc).__name__}: {exc}", flush=True)

frame_results = pd.DataFrame(frame_rows)
errors_df = pd.DataFrame(method_errors)
# Fair comparison: aggregate only over frames that EVERY method processed successfully.
ok_frames = [s for s in kept_stems if all(s in frame_confs[m] for m in ENHANCERS)]
dropped = [s for s in kept_stems if s not in ok_frames]
if not ok_frames:
    raise RuntimeError("No frame was processed successfully by all methods; see errors_df above.")
if dropped:
    print(f"WARNING: {len(dropped)} frame(s) excluded from ALL aggregates because at least one method failed: {dropped}")
all_confusions = {m: sum(frame_confs[m][s] for s in ok_frames) for m in ENHANCERS}
print(f"Aggregates use {len(ok_frames)} frame(s) common to all {len(ENHANCERS)} variants.")

print("\nCompleted.")
print(f"Successful frame-method runs: {(frame_results['status'] == 'ok').sum()} / {len(frame_results)}")
if not errors_df.empty:
    display(errors_df)

## Aggregate results and compare methods

The primary comparison is aggregate performance across all labelled pixels from all successfully evaluated frames. The per-frame table is also saved so variability across frames is visible. `mIoU_present_classes` averages only classes with at least one ground-truth pixel in the evaluated set; classes absent from the annotations are not included in this mean.

In [ ]:
aggregate_rows = []
per_class_tables = {}
for method, conf in all_confusions.items():
    per_class, summary = metrics_from_confusion(conf)
    per_class_tables[method] = per_class
    aggregate_rows.append({"method": method, "frames": len(ok_frames), **summary})

aggregate_results = pd.DataFrame(aggregate_rows).set_index("method").sort_values(
    "mIoU_present_classes", ascending=False
)
display(aggregate_results.style.format({
    "mIoU_present_classes": "{:.4f}",
    "labelled_pixel_accuracy": "{:.4f}",
    "labelled_pixels": "{:,}",
    "classes_present": "{:.0f}",
    "frames": "{:.0f}",
}, na_rep="n/a"))

per_class_iou = pd.concat({m: t["iou"] for m, t in per_class_tables.items()}, axis=1)
per_class_recall = pd.concat({m: t["recall"] for m, t in per_class_tables.items()}, axis=1)
per_class_precision = pd.concat({m: t["precision"] for m, t in per_class_tables.items()}, axis=1)

print("Per-class IoU")
display(per_class_iou.style.format("{:.3f}", na_rep="n/a"))
print("Per-class recall")
display(per_class_recall.style.format("{:.3f}", na_rep="n/a"))
print("Per-class precision")
display(per_class_precision.style.format("{:.3f}", na_rep="n/a"))

ax = aggregate_results["mIoU_present_classes"].sort_values().plot.barh(figsize=(8, 4), xlim=(0, 1))
ax.set_xlabel("Mean IoU over classes present in GT")
ax.set_title("CoralScapes segmentation performance by input enhancement")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

ax = aggregate_results["labelled_pixel_accuracy"].sort_values().plot.barh(figsize=(8, 4), xlim=(0, 1))
ax.set_xlabel("Labelled-pixel accuracy")
ax.set_title("Labelled-pixel accuracy by input enhancement")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Paired bootstrap over frames

Each method is compared with **raw** on the *same* resampled frames (2,000 resamples by default), which cancels out which frames happen to be easy or hard. The 95% interval is the range of the metric difference across resamples; if it contains 0, the data cannot tell the method from raw. `Sea-thru − raw@640px` isolates the effect of Sea-thru's colour correction from the effect of its downscaling.

Caveats: frames are the resampling unit, but adjacent video frames are correlated, so the intervals are still optimistic. Mean recall and mIoU are the more reliable metrics; precision-like numbers are inflated by partial annotation.

In [ ]:
def fast_metrics(conf):
    """conf: (..., K, K+1) -> dict of arrays: mIoU / mean recall over classes present in GT, labelled-pixel accuracy."""
    conf = np.asarray(conf, dtype=np.float64)
    tp = np.einsum("...ii->...i", conf[..., :N_CLASSES])
    gt = conf.sum(-1)
    pr = conf[..., :N_CLASSES].sum(-2)
    present = gt > 0
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = tp / (gt + pr - tp)
        rec = tp / gt
    n_present = np.maximum(present.sum(-1), 1)
    return {"mIoU": np.where(present, iou, 0).sum(-1) / n_present,
            "mean_recall": np.where(present, rec, 0).sum(-1) / n_present,
            "pixel_acc": tp.sum(-1) / np.maximum(gt.sum(-1), 1)}

methods = list(ENHANCERS)
stack = {m: np.stack([frame_confs[m][s] for s in ok_frames]) for m in methods}     # (frames, K, K+1)
rng = np.random.default_rng(0)
boot_idx = rng.integers(0, len(ok_frames), size=(N_BOOT, len(ok_frames)))          # same resample for every method
boot = {m: fast_metrics(stack[m][boot_idx].sum(axis=1)) for m in methods}
point = {m: fast_metrics(stack[m].sum(axis=0)) for m in methods}

comparisons = [(m, "raw") for m in methods if m != "raw"]
if CONTROL_NAME in ENHANCERS and "Sea-thru" in ENHANCERS:
    comparisons.append(("Sea-thru", CONTROL_NAME))

rows = []
for a, b in comparisons:
    for metric in ("mIoU", "mean_recall", "pixel_acc"):
        d = boot[a][metric] - boot[b][metric]
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"comparison": f"{a} − {b}", "metric": metric,
                     "difference": float(point[a][metric] - point[b][metric]),
                     "ci95_low": float(lo), "ci95_high": float(hi),
                     "P(better)": float(np.mean(d > 0)),
                     "verdict": "better" if lo > 0 else "worse" if hi < 0 else "inconclusive"})
paired = pd.DataFrame(rows)
display(paired.style.format({"difference": "{:+.4f}", "ci95_low": "{:+.4f}", "ci95_high": "{:+.4f}", "P(better)": "{:.2f}"}))

sub_m = paired[paired.metric == "mIoU"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 0.55 * len(sub_m) + 1.6))
y = np.arange(len(sub_m))
ax.errorbar(sub_m["difference"], y,
            xerr=[sub_m["difference"] - sub_m["ci95_low"], sub_m["ci95_high"] - sub_m["difference"]],
            fmt="o", capsize=4)
ax.axvline(0, color="k", lw=1)
ax.set_yticks(y); ax.set_yticklabels(sub_m["comparison"]); ax.invert_yaxis()
ax.set_xlabel("Δ mIoU (95% bootstrap interval over frames)  —  right of 0 = better")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Based on {len(ok_frames)} frames, {N_BOOT} resamples.")

## Visual inspection of predictions

The prediction overlays are shown alongside the original image and ground truth. Enhancement can improve colour and contrast while also changing texture or edges in ways that hurt segmentation, so the metrics should be read together with the visual examples.

In [ ]:
def colourise_project_mask(mask):
    # Project palette is configured in configs/config.yaml.
    colours = np.asarray(cfg["classes"]["colours"], dtype=np.uint8)
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for i, colour in enumerate(colours):
        out[mask == i] = colour
    out[mask == UNMAPPED] = (100, 100, 100)
    return out

def overlay_mask(image_rgb, mask, alpha=0.45):
    image_rgb = np.asarray(image_rgb).astype(np.float32)
    colour = colourise_project_mask(mask).astype(np.float32)
    valid = mask != IGNORE_INDEX
    out = image_rgb.copy()
    out[valid] = (1-alpha) * out[valid] + alpha * colour[valid]
    return np.clip(out, 0, 255).astype(np.uint8)

# Select first few frames for visual review. Each row is a frame; columns are
# original, ground truth overlay, then the mapped prediction for each method.
n_show = min(3, len(kept_stems))
for stem in kept_stems[:n_show]:
    original = np.asarray(load_rgb(stem))
    gt = load_gt(stem)
    methods_available = []
    for method in ENHANCERS:
        path = pred_dir / f"{stem}__{safe(method)}.png"
        if path.exists():
            methods_available.append((method, np.asarray(Image.open(path))))
    fig, axes = plt.subplots(1, 2 + len(methods_available), figsize=(4 * (2 + len(methods_available)), 4))
    axes[0].imshow(original); axes[0].set_title("Raw frame")
    axes[1].imshow(overlay_mask(original, gt)); axes[1].set_title("CVAT ground truth")
    for ax, (method, pred) in zip(axes[2:], methods_available):
        ax.imshow(overlay_mask(original, pred)); ax.set_title(method)
    for ax in axes:
        ax.axis("off")
    plt.suptitle(stem)
    plt.tight_layout()
    plt.show()

## Save tables and confusion matrices

The CSV files are written to `work/enhancement_comparison/`. They make the comparison easier to report and reproduce.

In [ ]:
aggregate_results.to_csv(OUTPUT_DIR / "aggregate_metrics.csv")
paired.to_csv(OUTPUT_DIR / "paired_bootstrap.csv", index=False)
frame_results.to_csv(OUTPUT_DIR / "per_frame_metrics.csv", index=False)
per_class_iou.to_csv(OUTPUT_DIR / "per_class_iou.csv")
per_class_recall.to_csv(OUTPUT_DIR / "per_class_recall.csv")
per_class_precision.to_csv(OUTPUT_DIR / "per_class_precision.csv")
for method, conf in all_confusions.items():
    pd.DataFrame(conf, index=PROJECT_CLASSES,
                 columns=PROJECT_CLASSES + ["unmapped"]).to_csv(OUTPUT_DIR / f"confusion_{safe(method)}.csv")
if not errors_df.empty:
    errors_df.to_csv(OUTPUT_DIR / "errors.csv", index=False)

print("Saved comparison outputs to:", OUTPUT_DIR)
print("\nAggregate ranking (higher is better except runtime):")
display(aggregate_results)

## Notes for interpreting the results

- **Raw passthrough is the baseline.** Every enhancement method should be compared with this result, not only with the other enhanced inputs.
- **This is not a general test of enhancement across all reef types.** It measures how these methods affect this Coralscapes model on this small annotated Moreton Bay dataset.
- **The mapping matters.** If the CVAT labels include morphologies that Coralscapes does not predict directly, those categories can have low or undefined scores. The mapping in `configs/config.yaml` should only be changed where the semantic correspondence is defensible.
- **Sea-thru assumptions matter.** The utility estimates relative depth and maps it to an assumed range, rather than using measured range. Its scores are exploratory, and the parameters should be recorded when reporting results.
- **Runtimes are not a hardware benchmark.** Device, caching, image resolution and warm-up are not controlled here. The times are included mainly to estimate practical runtime in this notebook.
- **Small samples need caution.** If annotated frames are adjacent or highly similar, frame-level metrics are not independent. A held-out set of spatially or temporally separated clips is preferable when enough annotations exist.
- **The paired bootstrap matters more than the ranking table.** A method only counts as improving performance if its interval versus raw excludes 0. With a handful of frames most differences are inconclusive.
- **The resolution control needs checking.** If `Sea-thru − raw@640px` is near 0 but `Sea-thru − raw` is negative, the loss comes from downscaling, not from the colour correction. FUnIE-GAN also works at a small fixed size (256 px), so a similar control for it would be a useful addition.
- **Everything is scored through the class mapping.** Changing `configs/config.yaml` (for example mapping `other coral *`) changes the baseline for all variants at once.